# Sección · Título

**Autor:** _nombre_ · **Fecha:** _dd/mm/2026_

> **Qué responde:** las preguntas analíticas que cubre este notebook.

Plantilla de la Fase 3 para prototipar en tu compu (CSV de 2025) con el mismo código que corre en la instancia de AWS (datos en HDFS): **solo cambia `LOCAL`**. Lo que salga de aquí se copia después al notebook final `fase3_pipeline.ipynb`.

**Antes de empezar:** copia este archivo con un nombre propio, por ejemplo `C_visualizaciones.ipynb`. La instalación y el flujo de trabajo están en `notebooks/README.md`.

## 1. Configuración

In [ ]:
LOCAL = True          # True: tu compu con el CSV de 2025 · False: la instancia con el histórico completo
PARTE = "C"           # sección del notebook (A, B o C); se usa en los nombres de las figuras

CSV_LOCAL = "../data/Crimes_2025.csv"   # ver notebooks/README.md para descargarlo
TABLA_CLUSTER = "chicago.crimes"        # capa processed en HDFS, registrada en el catálogo Hive

FUENTE = "CSV 2025 (local)" if LOCAL else "Histórico completo (HDFS)"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import SparkSession, functions as F

CARPETA_FIGURAS = Path("figuras")
CARPETA_FIGURAS.mkdir(exist_ok=True)


def guardar(fig, nombre):
    """Guarda la figura como PNG en notebooks/figuras/ para el reporte."""
    sufijo = "local" if LOCAL else "cluster"
    ruta = CARPETA_FIGURAS / f"{PARTE}_{nombre}_{sufijo}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Figura guardada en", ruta)

## 2. Sesión de Spark y carga de datos

`cargar_crimenes()` devuelve **las mismas columnas** en los dos modos, así que el resto del notebook no cambia:

| Columna | Tipo | Nota |
|---|---|---|
| `id`, `case_number` | long, string | Identificadores |
| `fecha` | timestamp | Fecha y hora del incidente |
| `anio`, `mes`, `hora` | int | Derivadas de `fecha` |
| `dia_semana` | int | 1 = domingo … 7 = sábado |
| `primary_type`, `description` | string | Tipo de delito |
| `location_description` | string | `DESCONOCIDO` si falta |
| `arrest`, `domestic` | boolean | |
| `beat`, `district`, `ward`, `community_area` | int | Niveles geográficos |
| `latitude`, `longitude`, `x_coordinate`, `y_coordinate` | double | Coordenadas |
| `hora_imputada` | boolean | Hora exacta 00:00:00 o 12:00:00 (artefacto de captura) |
| `coord_imputada` | boolean | Sin coordenadas originales |

In [ ]:
builder = (SparkSession.builder
           .appName(f"fase3-{PARTE}-{'local' if LOCAL else 'cluster'}")
           .config("spark.sql.session.timeZone", "UTC"))

if LOCAL:
    spark = (builder.master("local[*]")
             .config("spark.driver.memory", "2g")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
else:
    spark = builder.enableHiveSupport().getOrCreate()   # master = yarn (spark-defaults.conf)

spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master, "|", FUENTE)

In [ ]:
def cargar_crimenes():
    """Incidentes con el esquema de la capa processed (ver tabla de arriba)."""
    if not LOCAL:
        return spark.table(TABLA_CLUSTER)

    # En local se replica, de forma simplificada, la limpieza de la capa processed
    raw = spark.read.csv(CSV_LOCAL, header=True)
    fecha = F.try_to_timestamp(F.col("Date"), F.lit("MM/dd/yyyy hh:mm:ss a"))
    df = raw.select(
        F.col("ID").try_cast("long").alias("id"),
        F.col("Case Number").alias("case_number"),
        fecha.alias("fecha"),
        F.col("Primary Type").alias("primary_type"),
        F.col("Description").alias("description"),
        F.coalesce(F.col("Location Description"), F.lit("DESCONOCIDO")).alias("location_description"),
        F.col("Arrest").try_cast("boolean").alias("arrest"),
        F.col("Domestic").try_cast("boolean").alias("domestic"),
        F.col("Beat").try_cast("int").alias("beat"),
        F.col("District").try_cast("int").alias("district"),
        F.col("Ward").try_cast("int").alias("ward"),
        F.col("Community Area").try_cast("int").alias("community_area"),
        F.col("Latitude").try_cast("double").alias("latitude"),
        F.col("Longitude").try_cast("double").alias("longitude"),
        F.col("X Coordinate").try_cast("double").alias("x_coordinate"),
        F.col("Y Coordinate").try_cast("double").alias("y_coordinate"),
    )
    return (df
            .withColumn("anio", F.year("fecha"))
            .withColumn("mes", F.month("fecha"))
            .withColumn("dia_semana", F.dayofweek("fecha"))
            .withColumn("hora", F.hour("fecha"))
            .withColumn("hora_imputada",
                        F.hour("fecha").isin(0, 12) & (F.minute("fecha") == 0) & (F.second("fecha") == 0))
            .withColumn("coord_imputada", F.col("latitude").isNull()))


df = cargar_crimenes().cache()
print(f"{df.count():,} incidentes")
df.printSchema()

## 3. Pregunta analítica

- **Pregunta N:** escribe la pregunta tal como irá en el título de la gráfica.
- **Qué se espera ver:** …
- **Filtros:** p. ej. excluir `hora_imputada` en análisis horarios, o `coord_imputada` en análisis espaciales finos.

## 4. Análisis

**Regla:** agregar en Spark y traer a pandas **solo el resultado pequeño** con `.toPandas()`. Nunca `df.toPandas()` sobre el DataFrame completo: el histórico tiene 8.6 millones de filas y no cabe en memoria.

El ejemplo de abajo (incidentes por hora) es solo para mostrar el patrón; reemplázalo con tu análisis.

In [ ]:
por_hora = (df.filter(~F.col("hora_imputada"))
              .groupBy("hora").count()
              .orderBy("hora")
              .toPandas())
por_hora.head()

## 5. Gráficas

El título de cada gráfica empieza con "Pregunta N:" (lo pide la actividad). Cada gráfica se guarda con `guardar(fig, "nombre")` en `notebooks/figuras/` para usarla en el reporte.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(por_hora["hora"], por_hora["count"], color="#2a78d6")
ax.set(title=f"Pregunta N: ¿A qué hora del día ocurren más incidentes? · {FUENTE}",
       xlabel="Hora del día", ylabel="Incidentes")
ax.set_xticks(range(24))
ax.spines[["top", "right"]].set_visible(False)
guardar(fig, "incidentes_por_hora")
plt.show()

## 6. Interpretación

De 3 a 5 oraciones en lenguaje de negocio (lo pide la actividad):

- **Qué muestra la gráfica**, con las cifras clave.
- **Qué significa para el problema:** dónde o cuándo conviene poner patrullas.
- **Limitaciones:** qué podría sesgar el resultado (horas imputadas, sesgo de patrullaje, reportes tardíos…).

In [ ]:
spark.stop()   # libera la sesión: en la instancia solo cabe una sesión de Spark a la vez